# Binary Classification with a Hybrid CNN + Transformer

A small convolutional backbone (for data efficiency) feeds a transformer encoder
that applies self-attention over the feature-map tokens.

## Setup

In [1]:
import datetime

import numpy as np
import tensorflow as tf
from sklearn.metrics import precision_recall_curve

notebook_start_time = datetime.datetime.now()

I0000 00:00:1782381995.388032 1097750 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782381996.418938 1097750 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
test_dir = "data/test"
train_dir = "data/train"
validation_dir = "data/val"

height, width, channels = 150, 150, 1

batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(height, width),
    batch_size=batch_size,
    label_mode="int",
    color_mode="grayscale",
)

# validation/test are shuffle=False so labels stay aligned with model.predict
validation_ds = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=(height, width),
    batch_size=batch_size,
    label_mode="int",
    color_mode="grayscale",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(height, width),
    batch_size=batch_size,
    label_mode="int",
    color_mode="grayscale",
    shuffle=False,
)

y_true = np.concatenate([y for x, y in test_ds], axis=0)

Found 4442 files belonging to 2 classes.
Found 790 files belonging to 2 classes.
Found 624 files belonging to 2 classes.


In [3]:
from visualization import show_confusion_matrix, summary_graphics


def get_class_training_weights(train_ds, normalize=True):
    labels, counts = np.unique(
        np.concatenate([y for x, y in train_ds], axis=0), return_counts=True
    )
    total = sum(counts)

    weights = [total / (2 * count) for count in counts]
    max_weight = np.max(weights)

    if normalize:
        return {label: weights[i] / max_weight for i, label in enumerate(labels)}

    return {label: weights[i] for i, label in enumerate(labels)}

In [4]:
class_weight = get_class_training_weights(train_ds=train_ds)

print(f"Weight for normal class: {class_weight[0]:1.3f}")
print(f"Weight for pneumonia class: {class_weight[1]:1.3f}")

Weight for normal class: 1.000
Weight for pneumonia class: 0.348


## Model: a hybrid CNN + transformer

A pure Vision Transformer needs far more data than we have (~4.4k images), so we
keep a small **convolutional backbone** for its built-in locality/translation bias,
then treat its `9x9x128` feature map as a sequence of **81 tokens** and run a
**transformer encoder** (multi-head self-attention + MLP, each with a residual
connection and pre-norm, plus a learnable position embedding) so the model can
relate distant regions of the lung. We average the tokens and classify.

In [5]:
metrics = [
    tf.keras.metrics.TruePositives(name="tp"),
    tf.keras.metrics.TrueNegatives(name="tn"),
    tf.keras.metrics.BinaryAccuracy(name="accuracy"),
    tf.keras.metrics.Precision(name="precision"),
    tf.keras.metrics.Recall(name="recall"),
    tf.keras.metrics.AUC(name="auc"),
]


def make_callbacks(filepath):
    # select on val AUC (threshold-free, robust to the 74/26 class imbalance)
    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=filepath, save_best_only=True, monitor="val_auc", mode="max"
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc", mode="max", patience=5, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", mode="min", factor=0.5, patience=3
        ),
    ]


epochs = 30

In [6]:
@tf.keras.utils.register_keras_serializable()
class AddPositionEmbedding(tf.keras.layers.Layer):
    """Add a learnable position embedding to a sequence of tokens."""

    def __init__(self, num_tokens, proj_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_tokens = num_tokens
        self.proj_dim = proj_dim
        self.pos_emb = tf.keras.layers.Embedding(num_tokens, proj_dim)

    def build(self, input_shape):
        self.pos_emb.build((self.num_tokens,))
        super().build(input_shape)

    def call(self, x):
        positions = tf.range(start=0, limit=self.num_tokens, delta=1)
        return x + self.pos_emb(positions)

    def get_config(self):
        config = super().get_config()
        config.update(num_tokens=self.num_tokens, proj_dim=self.proj_dim)
        return config


def transformer_encoder(x, num_heads, key_dim, mlp_dim, dropout):
    # multi-head self-attention sublayer (pre-norm + residual)
    y = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    y = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=key_dim, dropout=dropout
    )(y, y)
    x = tf.keras.layers.Add()([x, y])
    # feed-forward sublayer (pre-norm + residual)
    y = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    y = tf.keras.layers.Dense(mlp_dim, activation="gelu")(y)
    y = tf.keras.layers.Dropout(dropout)(y)
    y = tf.keras.layers.Dense(x.shape[-1])(y)
    return tf.keras.layers.Add()([x, y])


def build_model(proj_dim=128, num_heads=4, num_blocks=2, dropout=0.1):
    inputs = tf.keras.Input((height, width, channels), name="input")

    # light augmentation (no horizontal flip -- anatomically wrong for chest X-rays)
    x = tf.keras.layers.RandomRotation(0.1)(inputs)
    x = tf.keras.layers.RandomTranslation(0.1, 0.1)(x)
    x = tf.keras.layers.Rescaling(1.0 / 255)(x)

    # convolutional backbone -- gives locality/translation bias on a small dataset
    x = tf.keras.layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(x)
    x = tf.keras.layers.SeparableConv2D(64, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.SeparableConv2D(proj_dim, 3, padding="same", activation="relu")(
        x
    )
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.SeparableConv2D(proj_dim, 3, padding="same", activation="relu")(
        x
    )
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    # treat the H x W x proj_dim feature map as a sequence of tokens for attention
    num_tokens = x.shape[1] * x.shape[2]
    x = tf.keras.layers.Reshape((num_tokens, proj_dim))(x)
    x = AddPositionEmbedding(num_tokens, proj_dim)(x)
    for _ in range(num_blocks):
        x = transformer_encoder(
            x,
            num_heads=num_heads,
            key_dim=proj_dim // num_heads,
            mlp_dim=proj_dim * 2,
            dropout=dropout,
        )

    # pool the tokens and classify
    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="output")(x)
    return tf.keras.Model(inputs, outputs, name="hybrid_cnn_transformer")

In [7]:
model_1 = build_model()
model_1.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=metrics,
)
model_1.summary()

Model: "hybrid_cnn_transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 150, 150,  │          0 │ -                 │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_rotation     │ (None, 150, 150,  │          0 │ input[0][0]       │
│ (RandomRotation)    │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_translation  │ (None, 150, 150,  │          0 │ random_rotation[… │
│ (RandomTranslation) │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 150, 150,  │          0 │ random_translati… │
│ (Rescaling)         │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 75, 75,    │        320 │ rescaling[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d    │ (None, 75, 75,    │      2,400 │ conv2d[0][0]      │
│ (SeparableConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 75, 75,    │        256 │ separable_conv2d… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 37, 37,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_1  │ (None, 37, 37,    │      8,896 │ max_pooling2d[0]… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 37, 37,    │        512 │ separable_conv2d… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 18, 18,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_2  │ (None, 18, 18,    │     17,664 │ max_pooling2d_1[… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 18, 18,    │        512 │ separable_conv2d… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 9, 9, 128) │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 81, 128)   │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_position_embed… │ (None, 81, 128)   │     10,368 │ reshape[0][0]     │
│ (AddPositionEmbedd… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 81, 128)   │        256 │ add_position_emb… │
│ (LayerNormalizatio… │                   │            │                 

 Total params: 306,273 (1.17 MB)

 Trainable params: 305,633 (1.17 MB)

 Non-trainable params: 640 (2.50 KB)

In [8]:
history = model_1.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=epochs,
    class_weight=class_weight,
    verbose=1,
    callbacks=make_callbacks("best_model.keras"),
)
best_model = tf.keras.models.load_model("best_model.keras")
summary_graphics(history, best_model, test_ds)

Epoch 1/30


/home/sam/Documents/projects/ConvolutedComputerVision/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


139/139 ━━━━━━━━━━━━━━━━━━━━ 56s 370ms/step - accuracy: 0.8222 - auc: 0.9119 - loss: 0.1918 - precision: 0.9507 - recall: 0.8019 - tn: 1009.0000 - tp: 2643.0000 - val_accuracy: 0.7430 - val_auc: 0.5642 - val_loss: 0.7357 - val_precision: 0.7430 - val_recall: 1.0000 - val_tn: 0.0000e+00 - val_tp: 587.0000 - learning_rate: 1.0000e-04
Epoch 2/30
139/139 ━━━━━━━━━━━━━━━━━━━━ 68s 487ms/step - accuracy: 0.8890 - auc: 0.9577 - loss: 0.1315 - precision: 0.9701 - recall: 0.8774 - tn: 1057.0000 - tp: 2892.0000 - val_accuracy: 0.7430 - val_auc: 0.4984 - val_loss: 1.0614 - val_precision: 0.7430 - val_recall: 1.0000 - val_tn: 0.0000e+00 - val_tp: 587.0000 - learning_rate: 1.0000e-04
Epoch 3/30
139/139 ━━━━━━━━━━━━━━━━━━━━ 46s 322ms/step - accuracy: 0.9192 - auc: 0.9764 - loss: 0.1001 - precision: 0.9779 - recall: 0.9117 - tn: 1078.0000 - tp: 3005.0000 - val_accuracy: 0.7430 - val_auc: 0.5230 - val_loss: 1.2508 - val_precision: 0.7430 - val_recall: 1.0000 - val_tn: 0.0000e+00 - val_tp: 587.0000 - le

KeyboardInterrupt: 

## Tuning the decision threshold

At the default 0.5 cutoff a high-precision/low-recall model produces many false
negatives. We pick the threshold that maximizes F1 on the **validation** set and
apply it to the **test** set.

In [ ]:
# Choose the decision threshold on the VALIDATION set (never on test), then apply it
# to test. A lower threshold trades a little precision for far fewer false negatives.
val_probs = best_model.predict(validation_ds).ravel()
y_val = np.concatenate([y for _, y in validation_ds], axis=0)

precision, recall, thresholds = precision_recall_curve(y_val, val_probs)
f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-9)
threshold = float(thresholds[np.argmax(f1)])
print(f"Tuned threshold: {threshold:.3f}  (default is 0.5)\n")

test_probs = best_model.predict(test_ds).ravel()
y_test = np.concatenate([y for _, y in test_ds], axis=0)

for name, thr in [("default 0.5", 0.5), (f"tuned {threshold:.3f}", threshold)]:
    pred = (test_probs >= thr).astype("int32")
    tp = int(((pred == 1) & (y_test == 1)).sum())
    fn = int(((pred == 0) & (y_test == 1)).sum())
    fp = int(((pred == 1) & (y_test == 0)).sum())
    print(
        f"{name:>14}: accuracy={np.mean(pred == y_test):.3f}  "
        f"recall={tp / (tp + fn):.3f}  precision={tp / (tp + fp):.3f}  "
        f"false negatives={fn}"
    )

show_confusion_matrix(
    y_test,
    (test_probs >= threshold).astype("int32"),
    title=f"Confusion Matrix (threshold = {threshold:.3f})",
)

In [ ]:
notebook_end_time = datetime.datetime.now()
print(
    f"Notebook last run (end-to-end): {notebook_end_time} "
    f"(duration: {notebook_end_time - notebook_start_time})"
)